# 03 — Download and stack

Downloads finished chunks from Cloud Storage (checksum + grid verification), builds the per-date VRT stacks and runs QA.

> **Safe to re-run:** finished work is skipped. If the kernel dies or a cell crashes, just run the same cells again.

## Setup

**Which config is used?** After a run is chosen, everything uses the run's **frozen** config (`runs/<run_id>/run_config.yaml`), so this run's files are always read with the settings that produced them. If you edited your own config since `new-run`, a warning lists the differing keys; such changes need a **new run**. `auth.project` also stays the run's project. Taken from your config instead: `qa.acknowledged_issues` (accepting QA issues is a decision made after the run), `auth.key_file` and `resources` (they describe the machine, so a run exported on a laptop can be continued on a cloud notebook server).

In [ ]:
from pathlib import Path
import sys

# Project root = parent of notebooks/. Adding src/ is only needed if you did not run `pip install -e .`
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

# >>> Change this to YOUR config file (copied from config/pipeline.example.yaml; it must live in config/) <<<
CONFIG_PATH = ROOT / "config" / "my_aoi_season.yaml"

from sar_pipeline import config, resources
user_cfg = config.load_config(ROOT / CONFIG_PATH)   # your editable config; the RUN config is loaded below
print("Config :", CONFIG_PATH)
print("Machine:", resources.detect_resources(user_cfg).describe())

from sar_pipeline.cli import load_run_context

RUN_ID = None   # None = newest run; or a run folder name such as "v001_20260915"
cfg, run_path = load_run_context(user_cfg, RUN_ID)   # frozen run config from here on
print("Run:", run_path)

## Step 6 — Size check

Nothing is downloaded here: this only lists files (once per track folder) and compares their size to free disk space.

In [ ]:
from sar_pipeline import download

est = download.estimate_download(cfg, run_path)
gb = 1024 ** 3
print(f"Files: {est['n_files']}  Size: {est['bytes'] / gb:.2f} GB  Free: {est['disk_free_bytes'] / gb:.2f} GB  Fits: {est['fits']}")
print("Already verified (skipped):", est["n_skipped_verified"])

## ⛔ Confirm the download

If the size is fine, **change `CONFIRMED` to `True`**. Each file is checksum-verified and checked against the grid, dtype and nodata value. Network/checksum errors are retried up to 3×; a file failing verification gets one fresh download. A correctly aligned file containing only nodata becomes `VERIFIED_EMPTY`. Only one download may run per run.

Options: `RETRY_FAILED = True` retries `FAILED_DOWNLOAD` rows; `DEEP = True` re-hashes every local file instead of trusting size + modification time.

In [ ]:
CONFIRMED = False     # <- set to True after checking the size above
RETRY_FAILED = False
DEEP = False

m = download.download_completed(cfg, run_path, confirmed=CONFIRMED, retry_failed=RETRY_FAILED, deep=DEEP)
m["state"].value_counts()

## Step 7 — Stack and QA

For every selected track: per-date VRTs, `stack_VV.vrt` / `stack_VH.vrt`, `dates.csv`, and QA. All tracks are built first; if QA finds issues (missing dates, low valid pixels, failed/unverified/empty chunks) one `DecisionRequired` is raised and `decisions_required.md` is shown below.

In [ ]:
from IPython.display import Markdown, display
from sar_pipeline import stack
from sar_pipeline.errors import DecisionRequired

track_ids = [t["track_id"] for t in cfg["s1"]["tracks"]]
try:
    outs = stack.build_stacks(cfg, run_path, track_ids)
    for track, out in outs.items():
        print("Stack ready:", track, out)
except DecisionRequired as exc:
    print("DecisionRequired:", exc)
    display(Markdown((run_path / "decisions_required.md").read_text()))

## ⛔ Checkpoint 4 — decisions

For each issue decide: re-export (new run), retry (`export.retry_failed` + monitor, or download with `RETRY_FAILED`), or accept. To accept, add its `issue_id` to `qa.acknowledged_issues` in **your** config, re-run the Setup cell (it re-reads your config) and re-run the stack cell. Accepted dates stay in the stack and are flagged in `dates.csv`.

In [ ]:
import pandas as pd

TRACK = cfg["s1"]["tracks"][0]["track_id"]
dates = pd.read_csv(run_path / "stack" / f"track_{TRACK}" / "dates.csv")
dates   # valid_pct_<POL>: inside the planned scope; aoi_planned_pct: share of the AOI planned; n_temporal_neighbors: fewer = noisier

## Pilot inspection (Checkpoint 2)

Open `stack/track_<id>/VH/VH_<date>.vrt` in QGIS and check alignment with fields, dB ranges (VH ≈ −25…−10, VV ≈ −18…−3 over farmland), smooth fields with sharp edges, and similar brightness on opposite slopes. Then use `04_pixel_explorer.ipynb` on a known field.